In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# CONFIG
# ============================================================

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

loads = range(1, 10)


# ============================================================
# THREE EXPERIMENTS
#
# Change these paths to your actual v6, v7 and v8 folders.
# ============================================================

experiment_dirs = {

    "v6": Path(
        "/home/hsd/workspace/ns3-load-balance/rtt_sym_8_hosts_v6(prob1)/analysis_results"
    ),

    "v7": Path(
        "/home/hsd/workspace/ns3-load-balance/rtt_sym_8_hosts_v7(prob1)/analysis_results"
    ),

    "v8": Path(
        "/home/hsd/workspace/ns3-load-balance/rtt_sym_8_hosts_v8(prob1)/analysis_results"
    )
}


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

save_dir = Path(
    "/home/hsd/workspace/ns3-load-balance/"
    "rtt_sym_8_hosts_average_v6_v7_v8(prob1)/analysis_results"
)

save_dir.mkdir(
    parents=True,
    exist_ok=True
)


graph_dir = save_dir / "graph"
graph_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# FLOW SIZE THRESHOLDS
# ============================================================

SMALL = 100 * 1024          # 100 KB
LARGE = 1 * 1024 * 1024     # 1 MB


# ============================================================
# BASELINE
# ============================================================

baseline = "ecmp"


# ============================================================
# ALGORITHMS
#
# Keep this identical to your current notebook.
# ============================================================

algorithms = [

    "conga",

    # "lrtt_true",
    # "lrtt_false",

    "weighted_true",
    "weighted_false",

    "random_true",
    "random_false",

    # "top2_true",
    # "top2_false",

    # "weighted",
    # "random",
    # "top2",
]


# ============================================================
# LABELS
# ============================================================

label_map = {

    "conga":
        "Conga",

    "lrtt_true":
        "Lowest RTT (Timeout)",

    "lrtt_false":
        "Lowest RTT (No Timeout)",

    "weighted":
        "Weighted ECMP",

    "random":
        "Power-of-2 Random",

    "top2":
        "Power-of-2 Top2",

    "weighted_true":
        "Weighted ECMP (Timeout)",

    "weighted_false":
        "Weighted ECMP (No Timeout)",

    "random_true":
        "Power-of-2 Random (Timeout)",

    "random_false":
        "Power-of-2 Random (No Timeout)",

    "top2_true":
        "Power-of-2 Top2 (Timeout)",

    "top2_false":
        "Power-of-2 Top2 (No Timeout)"
}


# ============================================================
# COLORS
# ============================================================

color_map = {

    "conga":
        "tab:blue",

    "lrtt_true":
        "tab:cyan",

    "lrtt_false":
        "tab:cyan",

    "weighted":
        "tab:purple",

    "random":
        "tab:gray",

    "top2":
        "tab:olive",

    "weighted_true":
        "tab:brown",

    "weighted_false":
        "tab:brown",

    "random_true":
        "tab:orange",

    "random_false":
        "tab:orange",

    "top2_true":
        "tab:gray",

    "top2_false":
        "tab:gray"
}


# ============================================================
# MARKERS
# ============================================================

marker_map = {

    "conga":
        "o",

    "lrtt_true":
        "1",

    "lrtt_false":
        "2",

    "weighted":
        "D",

    "random":
        "v",

    "top2":
        "P",

    "weighted_true":
        "D",

    "weighted_false":
        "d",

    "random_true":
        "^",

    "random_false":
        "v",

    "top2_true":
        "P",

    "top2_false":
        "X"
}


# ============================================================
# CHECK EXPERIMENT DIRECTORIES
# ============================================================

print("=" * 80)
print("EXPERIMENT DIRECTORIES")
print("=" * 80)

for name, path in experiment_dirs.items():

    if path.exists():

        print(f"{name}: OK")
        print(f"      {path}")

    else:

        print(f"{name}: MISSING")
        print(f"      {path}")

print()


# ============================================================
# FUNCTION:
# LOAD ONE EXPERIMENT
# ============================================================

def load_experiment_csv(
    experiment_name,
    benchmark,
    load
):

    directory = experiment_dirs[
        experiment_name
    ]

    csv_path = (
        directory /
        f"{benchmark}_load_{load}_full.csv"
    )

    if not csv_path.exists():

        print(
            f"Missing {experiment_name}: "
            f"{csv_path}"
        )

        return None

    df = pd.read_csv(
        csv_path
    )

    if df.empty:

        print(
            f"Empty {experiment_name}: "
            f"{csv_path}"
        )

        return None

    return df


# ============================================================
# CHECK REQUIRED COLUMNS
# ============================================================

required_algorithms = [
    baseline
] + algorithms


required_columns = [
    f"FCT(s)_{algo}"
    for algo in required_algorithms
]


# ============================================================
# MAIN ANALYSIS
# ============================================================

summary_results = {}


for benchmark in benchmarks:

    print()
    print("=" * 80)
    print(f"BENCHMARK: {benchmark}")
    print("=" * 80)

    load_vals = []

    small_results = {
        algo: []
        for algo in algorithms
    }

    large_results = {
        algo: []
        for algo in algorithms
    }


    # ========================================================
    # LOOP OVER LOADS
    # ========================================================

    for load in loads:

        print()
        print("-" * 80)
        print(f"LOAD: {load / 10:.1f}")
        print("-" * 80)


        # ====================================================
        # LOAD ALL THREE EXPERIMENTS
        # ====================================================

        experiment_data = {}

        for experiment_name in experiment_dirs:

            df = load_experiment_csv(
                experiment_name,
                benchmark,
                load
            )

            if df is not None:

                experiment_data[
                    experiment_name
                ] = df


        # ====================================================
        # REQUIRE ALL THREE EXPERIMENTS
        #
        # We do NOT replace a missing experiment with zero.
        #
        # If one of v6/v7/v8 is missing, this load is skipped.
        # ====================================================

        missing = [
            name
            for name in experiment_dirs
            if name not in experiment_data
        ]

        if missing:

            print(
                f"Skipping load {load / 10:.1f}: "
                f"missing {', '.join(missing)}"
            )

            continue


        # ====================================================
        # CHECK COLUMNS
        # ====================================================

        invalid = False

        for experiment_name, df in experiment_data.items():

            missing_columns = [
                col
                for col in required_columns
                if col not in df.columns
            ]

            if missing_columns:

                print(
                    f"{experiment_name} is missing:"
                )

                for col in missing_columns:

                    print(
                        f"    {col}"
                    )

                invalid = True


        if invalid:

            continue


        # ====================================================
        # CALCULATE EXPERIMENT-LEVEL AVERAGES
        #
        # IMPORTANT:
        #
        # We calculate the average FCT separately for
        # each experiment first.
        #
        # Then:
        #
        #       v6 + v7 + v8
        #       -------------
        #             3
        #
        # This prevents one experiment with more flows
        # from automatically receiving more weight.
        # ====================================================

        experiment_averages = {}


        for experiment_name, df in experiment_data.items():

            # ------------------------------------------------
            # Remove invalid ECMP values
            # ------------------------------------------------

            df = df[
                df[f"FCT(s)_{baseline}"] > 0
            ].copy()

            if df.empty:

                print(
                    f"{experiment_name}: "
                    "no valid flows after ECMP filtering."
                )

                invalid = True

                break


            # ------------------------------------------------
            # SMALL FLOWS
            # ------------------------------------------------

            small_df = df[
                df["flow_size"] < SMALL
            ].copy()


            # ------------------------------------------------
            # LARGE FLOWS
            # ------------------------------------------------

            large_df = df[
                df["flow_size"] > LARGE
            ].copy()


            experiment_averages[
                experiment_name
            ] = {

                "small": {},

                "large": {}
            }


            # ------------------------------------------------
            # SMALL
            # ------------------------------------------------

            for algo in required_algorithms:

                col = f"FCT(s)_{algo}"

                if small_df.empty:

                    value = np.nan

                else:

                    value = small_df[
                        col
                    ].mean()

                experiment_averages[
                    experiment_name
                ]["small"][algo] = value


            # ------------------------------------------------
            # LARGE
            # ------------------------------------------------

            for algo in required_algorithms:

                col = f"FCT(s)_{algo}"

                if large_df.empty:

                    value = np.nan

                else:

                    value = large_df[
                        col
                    ].mean()

                experiment_averages[
                    experiment_name
                ]["large"][algo] = value


        if invalid:

            continue


        # ====================================================
        # AVERAGE V6 + V7 + V8
        # ====================================================

        averaged_fct = {

            "small": {},

            "large": {}
        }


        for size_category in [
            "small",
            "large"
        ]:

            for algo in required_algorithms:

                values = [

                    experiment_averages[
                        experiment_name
                    ][size_category][algo]

                    for experiment_name
                    in experiment_dirs
                ]

                values = [
                    value
                    for value in values
                    if pd.notna(value)
                ]

                if len(values) == 3:

                    averaged_value = (
                        sum(values) / 3
                    )

                else:

                    averaged_value = np.nan

                averaged_fct[
                    size_category
                ][algo] = averaged_value


        # ====================================================
        # SAVE AVERAGED RAW VALUES
        # ====================================================

        raw_row = {
            "load": load / 10
        }


        for algo in required_algorithms:

            raw_row[
                f"FCT(s)_{algo}_small"
            ] = averaged_fct[
                "small"
            ][algo]

            raw_row[
                f"FCT(s)_{algo}_large"
            ] = averaged_fct[
                "large"
            ][algo]


        # ====================================================
        # NORMALIZATION
        #
        # IMPORTANT:
        #
        # Normalize AFTER averaging v6/v7/v8.
        # ====================================================

        baseline_small = averaged_fct[
            "small"
        ][baseline]

        baseline_large = averaged_fct[
            "large"
        ][baseline]


        for algo in algorithms:

            # ------------------------------------------------
            # SMALL
            # ------------------------------------------------

            if (
                pd.notna(
                    averaged_fct["small"][algo]
                )
                and pd.notna(baseline_small)
                and baseline_small > 0
            ):

                normalized_small = (
                    averaged_fct["small"][algo]
                    /
                    baseline_small
                )

            else:

                normalized_small = np.nan


            # ------------------------------------------------
            # LARGE
            # ------------------------------------------------

            if (
                pd.notna(
                    averaged_fct["large"][algo]
                )
                and pd.notna(baseline_large)
                and baseline_large > 0
            ):

                normalized_large = (
                    averaged_fct["large"][algo]
                    /
                    baseline_large
                )

            else:

                normalized_large = np.nan


            small_results[
                algo
            ].append(
                normalized_small
            )

            large_results[
                algo
            ].append(
                normalized_large
            )


            # ------------------------------------------------
            # Store normalized values
            # ------------------------------------------------

            raw_row[
                f"Normalized_{algo}_small"
            ] = normalized_small

            raw_row[
                f"Normalized_{algo}_large"
            ] = normalized_large


        # ====================================================
        # SAVE THIS LOAD
        # ====================================================

        raw_df = pd.DataFrame(
            [raw_row]
        )

        raw_csv = (
            save_dir /
            f"{benchmark}_load_{load}_average.csv"
        )

        raw_df.to_csv(
            raw_csv,
            index=False
        )


        # ====================================================
        # PRINT RESULTS
        # ====================================================

        print(
            f"ECMP small average: "
            f"{baseline_small:.6f} s"
        )

        print(
            f"ECMP large average: "
            f"{baseline_large:.6f} s"
        )

        print()

        print(
            "Normalized SMALL:"
        )

        for algo in algorithms:

            value = small_results[
                algo
            ][-1]

            print(
                f"  {label_map[algo]:35s}: "
                f"{value:.4f}"
            )


        print()

        print(
            "Normalized LARGE:"
        )

        for algo in algorithms:

            value = large_results[
                algo
            ][-1]

            print(
                f"  {label_map[algo]:35s}: "
                f"{value:.4f}"
            )


        load_vals.append(
            load / 10
        )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    summary_results[
        benchmark
    ] = {

        "loads":
            load_vals,

        "small":
            small_results,

        "large":
            large_results
    }


# ============================================================
# GENERATE NORMALIZED GRAPHS
# ============================================================

for benchmark in benchmarks:

    results = summary_results[
        benchmark
    ]

    load_vals = results[
        "loads"
    ]

    small_results = results[
        "small"
    ]

    large_results = results[
        "large"
    ]


    # ========================================================
    # SMALL FLOWS
    # ========================================================

    plt.figure(
        figsize=(10, 7),
        dpi=120
    )

    for algo in algorithms:

        plt.plot(

            load_vals,

            small_results[algo],

            marker=marker_map[algo],

            color=color_map[algo],

            linestyle=(
                "--"
                if algo.endswith("_false")
                else "-"
            ),

            linewidth=2,

            markersize=7,

            label=(
                f"{label_map[algo]} / ECMP"
            )
        )


    plt.axhline(
        y=1,
        linestyle="--",
        color="black",
        linewidth=1.5,
        label="ECMP baseline"
    )


    plt.xlabel(
        "Network Load"
    )

    plt.ylabel(
        "Normalized Average FCT"
    )

    plt.title(
        f"{benchmark}: "
        f"Average of v6, v7 and v8 - "
        f"Small Flows (<100 KB)"
    )

    plt.ylim(
        0,
        1.8
    )

    plt.yticks(
        np.arange(
            0,
            1.81,
            0.2
        )
    )

    plt.grid(
        True,
        alpha=0.3
    )

    plt.legend()

    plt.tight_layout()


    out_path = (
        graph_dir /
        f"{benchmark}_SMALL_average_v6_v7_v8_relative_vs_load.png"
    )

    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"Saved: {out_path}"
    )


    # ========================================================
    # LARGE FLOWS
    # ========================================================

    plt.figure(
        figsize=(10, 7),
        dpi=120
    )

    for algo in algorithms:

        plt.plot(

            load_vals,

            large_results[algo],

            marker=marker_map[algo],

            color=color_map[algo],

            linestyle=(
                "--"
                if algo.endswith("_false")
                else "-"
            ),

            linewidth=2,

            markersize=7,

            label=(
                f"{label_map[algo]} / ECMP"
            )
        )


    plt.axhline(
        y=1,
        linestyle="--",
        color="black",
        linewidth=1.5,
        label="ECMP baseline"
    )


    plt.xlabel(
        "Network Load"
    )

    plt.ylabel(
        "Normalized Average FCT"
    )

    plt.title(
        f"{benchmark}: "
        f"Average of v6, v7 and v8 - "
        f"Large Flows (>1 MB)"
    )

    plt.ylim(
        0,
        1.4
    )

    plt.yticks(
        np.arange(
            0,
            1.41,
            0.2
        )
    )

    plt.grid(
        True,
        alpha=0.3
    )

    plt.legend()

    plt.tight_layout()


    out_path = (
        graph_dir /
        f"{benchmark}_LARGE_average_v6_v7_v8_relative_vs_load.png"
    )

    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"Saved: {out_path}"
    )


# ============================================================
# DONE
# ============================================================

print()
print("=" * 80)
print("DONE")
print("=" * 80)
print()
print(
    "Averaged v6 + v7 + v8 results generated."
)
print(
    f"Output directory: {save_dir}"
)

EXPERIMENT DIRECTORIES
v6: OK
      /home/hsd/workspace/ns3-load-balance/rtt_sym_8_hosts_v6(prob1)/analysis_results
v7: OK
      /home/hsd/workspace/ns3-load-balance/rtt_sym_8_hosts_v7(prob1)/analysis_results
v8: OK
      /home/hsd/workspace/ns3-load-balance/rtt_sym_8_hosts_v8(prob1)/analysis_results


BENCHMARK: private_enterprise

--------------------------------------------------------------------------------
LOAD: 0.1
--------------------------------------------------------------------------------
ECMP small average: 0.003790 s
ECMP large average: 0.353068 s

Normalized SMALL:
  Conga                              : 0.9883
  Weighted ECMP (Timeout)            : 0.9998
  Weighted ECMP (No Timeout)         : 0.9989
  Power-of-2 Random (Timeout)        : 0.9992
  Power-of-2 Random (No Timeout)     : 0.9980

Normalized LARGE:
  Conga                              : 0.9997
  Weighted ECMP (Timeout)            : 1.0002
  Weighted ECMP (No Timeout)         : 0.9997
  Power-of-2 Random (Time

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# CONFIG
# ============================================================

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

loads = range(1, 10)


# ============================================================
# THREE EXPERIMENTS
# ============================================================

experiment_dirs = {

    "v6": Path(
        "/home/hsd/workspace/ns3-load-balance/"
        "rtt_sym_8_hosts_v6(prob1)/analysis_results"
    ),

    "v7": Path(
        "/home/hsd/workspace/ns3-load-balance/"
        "rtt_sym_8_hosts_v7(prob1)/analysis_results"
    ),

    "v8": Path(
        "/home/hsd/workspace/ns3-load-balance/"
        "rtt_sym_8_hosts_v8(prob1)/analysis_results"
    )
}


# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

save_dir = Path(
    "/home/hsd/workspace/ns3-load-balance/"
    "rtt_sym_8_hosts_average_v6_v7_v8(prob1)/analysis_results"
)

save_dir.mkdir(
    parents=True,
    exist_ok=True
)


# Normalized graphs
graph_dir = save_dir / "graph"
graph_dir.mkdir(
    parents=True,
    exist_ok=True
)


# Raw average graphs
raw_graph_dir = save_dir / "raw_graph"
raw_graph_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# FLOW SIZE THRESHOLDS
# ============================================================

SMALL = 100 * 1024          # 100 KB
LARGE = 1 * 1024 * 1024     # 1 MB


# ============================================================
# BASELINE
# ============================================================

baseline = "ecmp"


# ============================================================
# ALGORITHMS
#
# Keep these names identical to the column names in your CSVs.
# ============================================================

algorithms = [

    "conga",

    # "lrtt_true",
    # "lrtt_false",

    "weighted_true",
    "weighted_false",

    "random_true",
    "random_false",

    # "top2_true",
    # "top2_false",

    # "weighted",
    # "random",
    # "top2",
]


# ============================================================
# LABELS
# ============================================================

label_map = {

    "ecmp":
        "ECMP",

    "conga":
        "Conga",

    "lrtt_true":
        "Lowest RTT (Timeout)",

    "lrtt_false":
        "Lowest RTT (No Timeout)",

    "weighted":
        "Weighted ECMP",

    "random":
        "Power-of-2 Random",

    "top2":
        "Power-of-2 Top2",

    "weighted_true":
        "Weighted ECMP (Timeout)",

    "weighted_false":
        "Weighted ECMP (No Timeout)",

    "random_true":
        "Power-of-2 Random (Timeout)",

    "random_false":
        "Power-of-2 Random (No Timeout)",

    "top2_true":
        "Power-of-2 Top2 (Timeout)",

    "top2_false":
        "Power-of-2 Top2 (No Timeout)"
}


# ============================================================
# COLORS
# ============================================================

color_map = {

    "ecmp":
        "black",

    "conga":
        "tab:blue",

    "lrtt_true":
        "tab:cyan",

    "lrtt_false":
        "tab:cyan",

    "weighted":
        "tab:purple",

    "random":
        "tab:gray",

    "top2":
        "tab:olive",

    "weighted_true":
        "tab:brown",

    "weighted_false":
        "tab:brown",

    "random_true":
        "tab:orange",

    "random_false":
        "tab:orange",

    "top2_true":
        "tab:gray",

    "top2_false":
        "tab:gray"
}


# ============================================================
# MARKERS
# ============================================================

marker_map = {

    "ecmp":
        "s",

    "conga":
        "o",

    "lrtt_true":
        "1",

    "lrtt_false":
        "2",

    "weighted":
        "D",

    "random":
        "v",

    "top2":
        "P",

    "weighted_true":
        "D",

    "weighted_false":
        "d",

    "random_true":
        "^",

    "random_false":
        "v",

    "top2_true":
        "P",

    "top2_false":
        "X"
}


# ============================================================
# CHECK EXPERIMENT DIRECTORIES
# ============================================================

print("=" * 80)
print("EXPERIMENT DIRECTORIES")
print("=" * 80)

for name, path in experiment_dirs.items():

    if path.exists():

        print(f"{name}: OK")
        print(f"    {path}")

    else:

        print(f"{name}: MISSING")
        print(f"    {path}")

print()


# ============================================================
# LOAD ONE EXPERIMENT CSV
# ============================================================

def load_experiment_csv(
    experiment_name,
    benchmark,
    load
):

    directory = experiment_dirs[
        experiment_name
    ]

    csv_path = (
        directory /
        f"{benchmark}_load_{load}_full.csv"
    )

    if not csv_path.exists():

        print(
            f"Missing {experiment_name}: "
            f"{csv_path}"
        )

        return None

    df = pd.read_csv(
        csv_path
    )

    if df.empty:

        print(
            f"Empty {experiment_name}: "
            f"{csv_path}"
        )

        return None

    return df


# ============================================================
# REQUIRED COLUMNS
# ============================================================

required_algorithms = [
    baseline
] + algorithms


required_columns = [
    f"FCT(s)_{algo}"
    for algo in required_algorithms
]


# ============================================================
# STORAGE FOR ALL RESULTS
# ============================================================

summary_results = {}


# ============================================================
# MAIN ANALYSIS
# ============================================================

for benchmark in benchmarks:

    print()
    print("=" * 80)
    print(f"BENCHMARK: {benchmark}")
    print("=" * 80)


    load_vals = []


    # --------------------------------------------------------
    # Normalized results
    # --------------------------------------------------------

    small_results = {
        algo: []
        for algo in algorithms
    }

    large_results = {
        algo: []
        for algo in algorithms
    }


    # --------------------------------------------------------
    # Raw average results
    # --------------------------------------------------------

    small_raw_results = {
        algo: []
        for algo in required_algorithms
    }

    large_raw_results = {
        algo: []
        for algo in required_algorithms
    }


    # ========================================================
    # LOOP OVER LOADS
    # ========================================================

    for load in loads:

        print()
        print("-" * 80)
        print(f"LOAD: {load / 10:.1f}")
        print("-" * 80)


        # ====================================================
        # LOAD V6, V7, V8
        # ====================================================

        experiment_data = {}


        for experiment_name in experiment_dirs:

            df = load_experiment_csv(
                experiment_name,
                benchmark,
                load
            )

            if df is not None:

                experiment_data[
                    experiment_name
                ] = df


        # ====================================================
        # REQUIRE ALL THREE EXPERIMENTS
        # ====================================================

        missing = [

            name

            for name in experiment_dirs

            if name not in experiment_data
        ]


        if missing:

            print(
                f"Skipping load {load / 10:.1f}: "
                f"missing {', '.join(missing)}"
            )

            continue


        # ====================================================
        # CHECK REQUIRED COLUMNS
        # ====================================================

        invalid = False


        for experiment_name, df in experiment_data.items():

            missing_columns = [

                col

                for col in required_columns

                if col not in df.columns
            ]


            if missing_columns:

                print(
                    f"{experiment_name} is missing:"
                )

                for col in missing_columns:

                    print(
                        f"    {col}"
                    )

                invalid = True


        if invalid:

            continue


        # ====================================================
        # EXPERIMENT-LEVEL AVERAGES
        #
        # Each experiment gets equal weight.
        #
        # First:
        #
        #   average FCT in v6
        #   average FCT in v7
        #   average FCT in v8
        #
        # Then:
        #
        #   (v6 + v7 + v8) / 3
        # ====================================================

        experiment_averages = {}


        for experiment_name, df in experiment_data.items():

            # ------------------------------------------------
            # Keep only valid ECMP flows
            # ------------------------------------------------

            df = df[
                df[f"FCT(s)_{baseline}"] > 0
            ].copy()


            if df.empty:

                print(
                    f"{experiment_name}: "
                    "no valid flows after ECMP filtering."
                )

                invalid = True

                break


            # ------------------------------------------------
            # SMALL FLOWS
            # ------------------------------------------------

            small_df = df[
                df["flow_size"] < SMALL
            ].copy()


            # ------------------------------------------------
            # LARGE FLOWS
            # ------------------------------------------------

            large_df = df[
                df["flow_size"] > LARGE
            ].copy()


            experiment_averages[
                experiment_name
            ] = {

                "small": {},

                "large": {}
            }


            # ------------------------------------------------
            # SMALL FLOW AVERAGES
            # ------------------------------------------------

            for algo in required_algorithms:

                col = f"FCT(s)_{algo}"


                if small_df.empty:

                    value = np.nan

                else:

                    value = small_df[
                        col
                    ].mean()


                experiment_averages[
                    experiment_name
                ][
                    "small"
                ][
                    algo
                ] = value


            # ------------------------------------------------
            # LARGE FLOW AVERAGES
            # ------------------------------------------------

            for algo in required_algorithms:

                col = f"FCT(s)_{algo}"


                if large_df.empty:

                    value = np.nan

                else:

                    value = large_df[
                        col
                    ].mean()


                experiment_averages[
                    experiment_name
                ][
                    "large"
                ][
                    algo
                ] = value


        if invalid:

            continue


        # ====================================================
        # AVERAGE V6 + V7 + V8
        # ====================================================

        averaged_fct = {

            "small": {},

            "large": {}
        }


        for size_category in [
            "small",
            "large"
        ]:

            for algo in required_algorithms:

                values = [

                    experiment_averages[
                        experiment_name
                    ][
                        size_category
                    ][
                        algo
                    ]

                    for experiment_name
                    in experiment_dirs
                ]


                # Only average when all three experiments
                # have a valid value.

                if all(
                    pd.notna(value)
                    for value in values
                ):

                    averaged_value = (
                        values[0]
                        +
                        values[1]
                        +
                        values[2]
                    ) / 3.0

                else:

                    averaged_value = np.nan


                averaged_fct[
                    size_category
                ][
                    algo
                ] = averaged_value


        # ====================================================
        # STORE RAW AVERAGES
        # ====================================================

        for algo in required_algorithms:

            small_raw_results[
                algo
            ].append(
                averaged_fct[
                    "small"
                ][
                    algo
                ]
            )


            large_raw_results[
                algo
            ].append(
                averaged_fct[
                    "large"
                ][
                    algo
                ]
            )


        # ====================================================
        # NORMALIZATION
        #
        # IMPORTANT:
        #
        # NORMALIZATION IS DONE AFTER THE V6/V7/V8
        # AVERAGE HAS BEEN CALCULATED.
        # ====================================================

        baseline_small = averaged_fct[
            "small"
        ][
            baseline
        ]


        baseline_large = averaged_fct[
            "large"
        ][
            baseline
        ]


        for algo in algorithms:

            # ------------------------------------------------
            # SMALL
            # ------------------------------------------------

            algorithm_small = averaged_fct[
                "small"
            ][
                algo
            ]


            if (
                pd.notna(algorithm_small)
                and
                pd.notna(baseline_small)
                and
                baseline_small > 0
            ):

                normalized_small = (
                    algorithm_small /
                    baseline_small
                )

            else:

                normalized_small = np.nan


            # ------------------------------------------------
            # LARGE
            # ------------------------------------------------

            algorithm_large = averaged_fct[
                "large"
            ][
                algo
            ]


            if (
                pd.notna(algorithm_large)
                and
                pd.notna(baseline_large)
                and
                baseline_large > 0
            ):

                normalized_large = (
                    algorithm_large /
                    baseline_large
                )

            else:

                normalized_large = np.nan


            small_results[
                algo
            ].append(
                normalized_small
            )


            large_results[
                algo
            ].append(
                normalized_large
            )


        # ====================================================
        # SAVE THIS LOAD AS CSV
        # ====================================================

        raw_row = {

            "load":
                load / 10
        }


        # ----------------------------------------------------
        # Raw FCT
        # ----------------------------------------------------

        for algo in required_algorithms:

            raw_row[
                f"FCT(s)_{algo}_small"
            ] = averaged_fct[
                "small"
            ][
                algo
            ]


            raw_row[
                f"FCT(s)_{algo}_large"
            ] = averaged_fct[
                "large"
            ][
                algo
            ]


        # ----------------------------------------------------
        # Normalized FCT
        # ----------------------------------------------------

        for algo in algorithms:

            raw_row[
                f"Normalized_{algo}_small"
            ] = small_results[
                algo
            ][-1]


            raw_row[
                f"Normalized_{algo}_large"
            ] = large_results[
                algo
            ][-1]


        # ----------------------------------------------------
        # Write CSV
        # ----------------------------------------------------

        averaged_df = pd.DataFrame(
            [raw_row]
        )


        csv_path = (
            save_dir /
            f"{benchmark}_load_{load}_average.csv"
        )


        averaged_df.to_csv(
            csv_path,
            index=False
        )


        # ====================================================
        # PRINT RESULTS
        # ====================================================

        print(
            f"Average ECMP SMALL FCT: "
            f"{baseline_small:.6f} s"
        )

        print(
            f"Average ECMP LARGE FCT: "
            f"{baseline_large:.6f} s"
        )


        print()
        print("Normalized SMALL:")


        for algo in algorithms:

            value = small_results[
                algo
            ][-1]


            print(
                f"  {label_map.get(algo, algo):35s}: "
                f"{value:.4f}"
            )


        print()
        print("Normalized LARGE:")


        for algo in algorithms:

            value = large_results[
                algo
            ][-1]


            print(
                f"  {label_map.get(algo, algo):35s}: "
                f"{value:.4f}"
            )


        load_vals.append(
            load / 10
        )


    # ========================================================
    # STORE BENCHMARK RESULTS
    # ========================================================

    summary_results[
        benchmark
    ] = {

        "loads":
            load_vals,

        "small":
            small_results,

        "large":
            large_results,

        "small_raw":
            small_raw_results,

        "large_raw":
            large_raw_results
    }


# ============================================================
# RAW AVERAGE FCT GRAPHS
#
# Saved separately in:
#
#     raw_graph/
#
# ============================================================

print()
print("=" * 80)
print("GENERATING RAW AVERAGE FCT GRAPHS")
print("=" * 80)


for benchmark in benchmarks:

    results = summary_results[
        benchmark
    ]


    load_vals = results[
        "loads"
    ]


    small_raw_results = results[
        "small_raw"
    ]


    large_raw_results = results[
        "large_raw"
    ]


    # ========================================================
    # SMALL FLOWS
    # ========================================================

    plt.figure(
        figsize=(10, 7),
        dpi=120
    )


    for algo in required_algorithms:

        values = small_raw_results[
            algo
        ]


        plt.plot(

            load_vals,

            values,

            marker=marker_map.get(
                algo,
                "o"
            ),

            color=color_map.get(
                algo,
                None
            ),

            linestyle=(
                "--"
                if algo.endswith("_false")
                else "-"
            ),

            linewidth=2,

            markersize=7,

            label=label_map.get(
                algo,
                algo
            )
        )


    plt.xlabel(
        "Network Load"
    )


    plt.ylabel(
        "Average FCT (s)"
    )


    plt.title(
        f"{benchmark}: "
        f"Average FCT - Small Flows (<100 KB)"
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.legend()


    plt.tight_layout()


    out_path = (
        raw_graph_dir /
        f"{benchmark}_SMALL_average_FCT_v6_v7_v8.png"
    )


    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close()


    print(
        f"Saved: {out_path}"
    )


    # ========================================================
    # LARGE FLOWS
    # ========================================================

    plt.figure(
        figsize=(10, 7),
        dpi=120
    )


    for algo in required_algorithms:

        values = large_raw_results[
            algo
        ]


        plt.plot(

            load_vals,

            values,

            marker=marker_map.get(
                algo,
                "o"
            ),

            color=color_map.get(
                algo,
                None
            ),

            linestyle=(
                "--"
                if algo.endswith("_false")
                else "-"
            ),

            linewidth=2,

            markersize=7,

            label=label_map.get(
                algo,
                algo
            )
        )


    plt.xlabel(
        "Network Load"
    )


    plt.ylabel(
        "Average FCT (s)"
    )


    plt.title(
        f"{benchmark}: "
        f"Average FCT - Large Flows (>1 MB)"
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.legend()


    plt.tight_layout()


    out_path = (
        raw_graph_dir /
        f"{benchmark}_LARGE_average_FCT_v6_v7_v8.png"
    )


    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close()


    print(
        f"Saved: {out_path}"
    )


# ============================================================
# NORMALIZED GRAPHS
#
# Saved in:
#
#     graph/
#
# ============================================================

print()
print("=" * 80)
print("GENERATING NORMALIZED GRAPHS")
print("=" * 80)


for benchmark in benchmarks:

    results = summary_results[
        benchmark
    ]


    load_vals = results[
        "loads"
    ]


    small_results = results[
        "small"
    ]


    large_results = results[
        "large"
    ]


    # ========================================================
    # SMALL FLOWS
    # ========================================================

    plt.figure(
        figsize=(10, 7),
        dpi=120
    )


    for algo in algorithms:

        plt.plot(

            load_vals,

            small_results[
                algo
            ],

            marker=marker_map.get(
                algo,
                "o"
            ),

            color=color_map.get(
                algo,
                None
            ),

            linestyle=(
                "--"
                if algo.endswith("_false")
                else "-"
            ),

            linewidth=2,

            markersize=7,

            label=(
                f"{label_map.get(algo, algo)} / ECMP"
            )
        )


    plt.axhline(

        y=1,

        linestyle="--",

        color="black",

        linewidth=1.5,

        label="ECMP baseline"
    )


    plt.xlabel(
        "Network Load"
    )


    plt.ylabel(
        "Normalized Average FCT"
    )


    plt.title(
        f"{benchmark}: "
        f"Average of v6, v7 and v8 - "
        f"Small Flows (<100 KB)"
    )


    plt.ylim(
        0,
        1.8
    )


    plt.yticks(
        np.arange(
            0,
            1.81,
            0.2
        )
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.legend()


    plt.tight_layout()


    out_path = (
        graph_dir /
        f"{benchmark}_SMALL_average_v6_v7_v8_relative_vs_load.png"
    )


    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close()


    print(
        f"Saved: {out_path}"
    )


    # ========================================================
    # LARGE FLOWS
    # ========================================================

    plt.figure(
        figsize=(10, 7),
        dpi=120
    )


    for algo in algorithms:

        plt.plot(

            load_vals,

            large_results[
                algo
            ],

            marker=marker_map.get(
                algo,
                "o"
            ),

            color=color_map.get(
                algo,
                None
            ),

            linestyle=(
                "--"
                if algo.endswith("_false")
                else "-"
            ),

            linewidth=2,

            markersize=7,

            label=(
                f"{label_map.get(algo, algo)} / ECMP"
            )
        )


    plt.axhline(

        y=1,

        linestyle="--",

        color="black",

        linewidth=1.5,

        label="ECMP baseline"
    )


    plt.xlabel(
        "Network Load"
    )


    plt.ylabel(
        "Normalized Average FCT"
    )


    plt.title(
        f"{benchmark}: "
        f"Average of v6, v7 and v8 - "
        f"Large Flows (>1 MB)"
    )


    plt.ylim(
        0,
        1.4
    )


    plt.yticks(
        np.arange(
            0,
            1.41,
            0.2
        )
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.legend()


    plt.tight_layout()


    out_path = (
        graph_dir /
        f"{benchmark}_LARGE_average_v6_v7_v8_relative_vs_load.png"
    )


    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close()


    print(
        f"Saved: {out_path}"
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("DONE")
print("=" * 80)

print()
print("Averaged experiments:")
print("    v6")
print("    v7")
print("    v8")

print()
print("Raw averaged CSVs:")
print(f"    {save_dir}")

print()
print("Raw FCT graphs:")
print(f"    {raw_graph_dir}")

print()
print("Normalized graphs:")
print(f"    {graph_dir}")

print()
print("All processing completed successfully.")

EXPERIMENT DIRECTORIES
v6: OK
    /home/hsd/workspace/ns3-load-balance/rtt_sym_8_hosts_v6(prob1)/analysis_results
v7: OK
    /home/hsd/workspace/ns3-load-balance/rtt_sym_8_hosts_v7(prob1)/analysis_results
v8: OK
    /home/hsd/workspace/ns3-load-balance/rtt_sym_8_hosts_v8(prob1)/analysis_results


BENCHMARK: private_enterprise

--------------------------------------------------------------------------------
LOAD: 0.1
--------------------------------------------------------------------------------
Average ECMP SMALL FCT: 0.003790 s
Average ECMP LARGE FCT: 0.353068 s

Normalized SMALL:
  Conga                              : 0.9883
  Weighted ECMP (Timeout)            : 0.9998
  Weighted ECMP (No Timeout)         : 0.9989
  Power-of-2 Random (Timeout)        : 0.9992
  Power-of-2 Random (No Timeout)     : 0.9980

Normalized LARGE:
  Conga                              : 0.9997
  Weighted ECMP (Timeout)            : 1.0002
  Weighted ECMP (No Timeout)         : 0.9997
  Power-of-2 Random (Ti

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# CONFIG
# ============================================================

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

loads = range(1, 10)


# ============================================================
# THREE EXPERIMENTS
# ============================================================

experiment_dirs = {

    "v6": Path(
        "/home/hsd/workspace/ns3-load-balance/"
        "rtt_sym_8_hosts_v6(prob1)/analysis_results"
    ),

    "v7": Path(
        "/home/hsd/workspace/ns3-load-balance/"
        "rtt_sym_8_hosts_v7(prob1)/analysis_results"
    ),

    "v8": Path(
        "/home/hsd/workspace/ns3-load-balance/"
        "rtt_sym_8_hosts_v8(prob1)/analysis_results"
    )
}


# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

save_dir = Path(
    "/home/hsd/workspace/ns3-load-balance/"
    "rtt_sym_8_hosts_average_v6_v7_v8(prob1)/analysis_results"
)

save_dir.mkdir(
    parents=True,
    exist_ok=True
)


# Normalized graphs
graph_dir = save_dir / "graph"
graph_dir.mkdir(
    parents=True,
    exist_ok=True
)


# Raw average graphs
raw_graph_dir = save_dir / "raw_graph"
raw_graph_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# FLOW SIZE THRESHOLDS
# ============================================================

SMALL = 100 * 1024          # 100 KB
LARGE = 1 * 1024 * 1024     # 1 MB


# ============================================================
# BASELINE
# ============================================================

baseline = "ecmp"


# ============================================================
# ALGORITHMS
#
# These must match the suffixes used in:
#
#     FCT(s)_<algorithm>
#
# in your CSV files.
# ============================================================

algorithms = [

    "conga",

    # "lrtt_true",
    # "lrtt_false",

    "weighted_true",
    "weighted_false",

    "random_true",
    "random_false",

    # "top2_true",
    # "top2_false",

    # "weighted",
    # "random",
    # "top2",
]


# ============================================================
# LABELS
# ============================================================

label_map = {

    "ecmp":
        "ECMP",

    "conga":
        "Conga",

    "lrtt_true":
        "Lowest RTT (Timeout)",

    "lrtt_false":
        "Lowest RTT (No Timeout)",

    "weighted":
        "Weighted ECMP",

    "random":
        "Power-of-2 Random",

    "top2":
        "Power-of-2 Top2",

    "weighted_true":
        "Weighted ECMP (Timeout)",

    "weighted_false":
        "Weighted ECMP (No Timeout)",

    "random_true":
        "Power-of-2 Random (Timeout)",

    "random_false":
        "Power-of-2 Random (No Timeout)",

    "top2_true":
        "Power-of-2 Top2 (Timeout)",

    "top2_false":
        "Power-of-2 Top2 (No Timeout)"
}


# ============================================================
# COLORS
# ============================================================

color_map = {

    "ecmp":
        "black",

    "conga":
        "tab:blue",

    "lrtt_true":
        "tab:cyan",

    "lrtt_false":
        "tab:cyan",

    "weighted":
        "tab:purple",

    "random":
        "tab:gray",

    "top2":
        "tab:olive",

    "weighted_true":
        "tab:brown",

    "weighted_false":
        "tab:brown",

    "random_true":
        "tab:orange",

    "random_false":
        "tab:orange",

    "top2_true":
        "tab:gray",

    "top2_false":
        "tab:gray"
}


# ============================================================
# MARKERS
# ============================================================

marker_map = {

    "ecmp":
        "s",

    "conga":
        "o",

    "lrtt_true":
        "1",

    "lrtt_false":
        "2",

    "weighted":
        "D",

    "random":
        "v",

    "top2":
        "P",

    "weighted_true":
        "D",

    "weighted_false":
        "d",

    "random_true":
        "^",

    "random_false":
        "v",

    "top2_true":
        "P",

    "top2_false":
        "X"
}


# ============================================================
# REQUIRED ALGORITHMS / COLUMNS
# ============================================================

required_algorithms = [
    baseline
] + algorithms


required_columns = [
    f"FCT(s)_{algo}"
    for algo in required_algorithms
]


# ============================================================
# CHECK EXPERIMENT DIRECTORIES
# ============================================================

print("=" * 80)
print("EXPERIMENT DIRECTORIES")
print("=" * 80)

for name, path in experiment_dirs.items():

    if path.exists():

        print(f"{name}: OK")
        print(f"    {path}")

    else:

        print(f"{name}: MISSING")
        print(f"    {path}")

print()


# ============================================================
# LOAD ONE EXPERIMENT CSV
# ============================================================

def load_experiment_csv(
    experiment_name,
    benchmark,
    load
):

    directory = experiment_dirs[
        experiment_name
    ]

    csv_path = (
        directory /
        f"{benchmark}_load_{load}_full.csv"
    )

    if not csv_path.exists():

        print(
            f"Missing {experiment_name}: "
            f"{csv_path}"
        )

        return None


    df = pd.read_csv(
        csv_path
    )


    if df.empty:

        print(
            f"Empty {experiment_name}: "
            f"{csv_path}"
        )

        return None


    return df


# ============================================================
# STORAGE
# ============================================================

summary_results = {}


# ============================================================
# MAIN ANALYSIS
# ============================================================

for benchmark in benchmarks:

    print()
    print("=" * 80)
    print(f"BENCHMARK: {benchmark}")
    print("=" * 80)


    load_vals = []


    # --------------------------------------------------------
    # Raw pooled FCT
    # --------------------------------------------------------

    small_raw_results = {
        algo: []
        for algo in required_algorithms
    }

    large_raw_results = {
        algo: []
        for algo in required_algorithms
    }


    # --------------------------------------------------------
    # Normalized FCT
    # --------------------------------------------------------

    small_normalized_results = {
        algo: []
        for algo in algorithms
    }

    large_normalized_results = {
        algo: []
        for algo in algorithms
    }


    # ========================================================
    # LOOP OVER LOADS
    # ========================================================

    for load in loads:

        print()
        print("-" * 80)
        print(f"LOAD: {load / 10:.1f}")
        print("-" * 80)


        # ====================================================
        # LOAD V6, V7, V8
        # ====================================================

        experiment_data = {}


        for experiment_name in experiment_dirs:

            df = load_experiment_csv(
                experiment_name,
                benchmark,
                load
            )


            if df is not None:

                experiment_data[
                    experiment_name
                ] = df


        # ====================================================
        # REQUIRE ALL THREE EXPERIMENTS
        # ====================================================

        missing = [

            name

            for name in experiment_dirs

            if name not in experiment_data
        ]


        if missing:

            print(
                f"Skipping load {load / 10:.1f}: "
                f"missing {', '.join(missing)}"
            )

            continue


        # ====================================================
        # CHECK REQUIRED COLUMNS
        # ====================================================

        invalid = False


        for experiment_name, df in experiment_data.items():

            missing_columns = [

                col

                for col in required_columns

                if col not in df.columns
            ]


            if missing_columns:

                print(
                    f"{experiment_name} is missing:"
                )


                for col in missing_columns:

                    print(
                        f"    {col}"
                    )


                invalid = True


        if invalid:

            continue


        # ====================================================
        # CREATE POOLED DATASET
        #
        # THIS IS THE IMPORTANT PART.
        #
        # We concatenate the actual flows from v6, v7,
        # and v8.
        #
        # Therefore:
        #
        #     pooled mean =
        #
        #     sum(all FCT observations)
        #     -------------------------
        #     number of observations
        #
        # This automatically gives each flow equal weight.
        # ====================================================

        pooled_data = {}


        for size_category in [
            "small",
            "large"
        ]:

            pooled_data[
                size_category
            ] = []


        for experiment_name in experiment_dirs:

            df = experiment_data[
                experiment_name
            ].copy()


            # ------------------------------------------------
            # Keep only flows with valid ECMP FCT.
            #
            # This is consistent across all three experiments.
            # ------------------------------------------------

            df = df[
                pd.to_numeric(
                    df[f"FCT(s)_{baseline}"],
                    errors="coerce"
                ) > 0
            ].copy()


            # ------------------------------------------------
            # Make sure flow_size is numeric
            # ------------------------------------------------

            df["flow_size"] = pd.to_numeric(
                df["flow_size"],
                errors="coerce"
            )


            df = df[
                df["flow_size"].notna()
            ].copy()


            # ------------------------------------------------
            # SMALL FLOWS
            # ------------------------------------------------

            small_df = df[
                df["flow_size"] < SMALL
            ].copy()


            # ------------------------------------------------
            # LARGE FLOWS
            # ------------------------------------------------

            large_df = df[
                df["flow_size"] > LARGE
            ].copy()


            pooled_data[
                "small"
            ].append(
                small_df
            )


            pooled_data[
                "large"
            ].append(
                large_df
            )


            print(
                f"{experiment_name}: "
                f"{len(small_df)} small, "
                f"{len(large_df)} large"
            )


        # ====================================================
        # CONCATENATE V6 + V7 + V8
        # ====================================================

        pooled_small = pd.concat(
            pooled_data["small"],
            ignore_index=True
        )


        pooled_large = pd.concat(
            pooled_data["large"],
            ignore_index=True
        )


        print()
        print(
            f"Pooled SMALL flows: "
            f"{len(pooled_small)}"
        )


        print(
            f"Pooled LARGE flows: "
            f"{len(pooled_large)}"
        )


        # ====================================================
        # CALCULATE POOLED AVERAGE FCT
        # ====================================================

        pooled_fct = {

            "small": {},

            "large": {}
        }


        # ====================================================
        # SMALL
        # ====================================================

        for algo in required_algorithms:

            column = f"FCT(s)_{algo}"


            if pooled_small.empty:

                value = np.nan

            else:

                values = pd.to_numeric(
                    pooled_small[column],
                    errors="coerce"
                )


                values = values[
                    values > 0
                ]


                if len(values) == 0:

                    value = np.nan

                else:

                    value = values.mean()


            pooled_fct[
                "small"
            ][
                algo
            ] = value


        # ====================================================
        # LARGE
        # ====================================================

        for algo in required_algorithms:

            column = f"FCT(s)_{algo}"


            if pooled_large.empty:

                value = np.nan

            else:

                values = pd.to_numeric(
                    pooled_large[column],
                    errors="coerce"
                )


                values = values[
                    values > 0
                ]


                if len(values) == 0:

                    value = np.nan

                else:

                    value = values.mean()


            pooled_fct[
                "large"
            ][
                algo
            ] = value


        # ====================================================
        # STORE RAW POOLED VALUES
        # ====================================================

        for algo in required_algorithms:

            small_raw_results[
                algo
            ].append(
                pooled_fct[
                    "small"
                ][
                    algo
                ]
            )


            large_raw_results[
                algo
            ].append(
                pooled_fct[
                    "large"
                ][
                    algo
                ]
            )


        # ====================================================
        # NORMALIZATION
        #
        # IMPORTANT:
        #
        # First pool v6/v7/v8.
        #
        # Then calculate:
        #
        #     algorithm pooled FCT
        #     --------------------
        #        ECMP pooled FCT
        #
        # NOT:
        #
        #     average of three normalized values.
        # ====================================================

        baseline_small = pooled_fct[
            "small"
        ][
            baseline
        ]


        baseline_large = pooled_fct[
            "large"
        ][
            baseline
        ]


        for algo in algorithms:

            # ------------------------------------------------
            # SMALL
            # ------------------------------------------------

            algorithm_small = pooled_fct[
                "small"
            ][
                algo
            ]


            if (
                pd.notna(algorithm_small)
                and
                pd.notna(baseline_small)
                and
                baseline_small > 0
            ):

                normalized_small = (
                    algorithm_small /
                    baseline_small
                )

            else:

                normalized_small = np.nan


            # ------------------------------------------------
            # LARGE
            # ------------------------------------------------

            algorithm_large = pooled_fct[
                "large"
            ][
                algo
            ]


            if (
                pd.notna(algorithm_large)
                and
                pd.notna(baseline_large)
                and
                baseline_large > 0
            ):

                normalized_large = (
                    algorithm_large /
                    baseline_large
                )

            else:

                normalized_large = np.nan


            small_normalized_results[
                algo
            ].append(
                normalized_small
            )


            large_normalized_results[
                algo
            ].append(
                normalized_large
            )


        # ====================================================
        # SAVE LOAD-LEVEL CSV
        # ====================================================

        raw_row = {

            "load":
                load / 10,

            "small_flow_count":
                len(pooled_small),

            "large_flow_count":
                len(pooled_large)
        }


        # ----------------------------------------------------
        # RAW POOLED FCT
        # ----------------------------------------------------

        for algo in required_algorithms:

            raw_row[
                f"FCT(s)_{algo}_small"
            ] = pooled_fct[
                "small"
            ][
                algo
            ]


            raw_row[
                f"FCT(s)_{algo}_large"
            ] = pooled_fct[
                "large"
            ][
                algo
            ]


        # ----------------------------------------------------
        # NORMALIZED
        # ----------------------------------------------------

        for algo in algorithms:

            raw_row[
                f"Normalized_{algo}_small"
            ] = small_normalized_results[
                algo
            ][-1]


            raw_row[
                f"Normalized_{algo}_large"
            ] = large_normalized_results[
                algo
            ][-1]


        averaged_df = pd.DataFrame(
            [raw_row]
        )


        csv_path = (
            save_dir /
            f"{benchmark}_load_{load}_average.csv"
        )


        averaged_df.to_csv(
            csv_path,
            index=False
        )


        # ====================================================
        # PRINT RESULTS
        # ====================================================

        print()
        print(
            f"Pooled ECMP SMALL FCT: "
            f"{baseline_small:.6f} s"
        )


        print(
            f"Pooled ECMP LARGE FCT: "
            f"{baseline_large:.6f} s"
        )


        print()
        print("Normalized SMALL:")


        for algo in algorithms:

            value = small_normalized_results[
                algo
            ][-1]


            print(
                f"  {label_map.get(algo, algo):35s}: "
                f"{value:.4f}"
            )


        print()
        print("Normalized LARGE:")


        for algo in algorithms:

            value = large_normalized_results[
                algo
            ][-1]


            print(
                f"  {label_map.get(algo, algo):35s}: "
                f"{value:.4f}"
            )


        load_vals.append(
            load / 10
        )


    # ========================================================
    # SAVE BENCHMARK RESULTS
    # ========================================================

    summary_results[
        benchmark
    ] = {

        "loads":
            load_vals,

        "small_raw":
            small_raw_results,

        "large_raw":
            large_raw_results,

        "small_normalized":
            small_normalized_results,

        "large_normalized":
            large_normalized_results
    }


# ============================================================
# RAW AVERAGE FCT GRAPHS
#
# Saved separately:
#
#     raw_graph/
# ============================================================

print()
print("=" * 80)
print("GENERATING RAW POOLED FCT GRAPHS")
print("=" * 80)


for benchmark in benchmarks:

    results = summary_results[
        benchmark
    ]


    load_vals = results[
        "loads"
    ]


    small_raw_results = results[
        "small_raw"
    ]


    large_raw_results = results[
        "large_raw"
    ]


    # ========================================================
    # SMALL FLOWS
    # ========================================================

    plt.figure(
        figsize=(10, 7),
        dpi=120
    )


    for algo in required_algorithms:

        plt.plot(

            load_vals,

            small_raw_results[
                algo
            ],

            marker=marker_map.get(
                algo,
                "o"
            ),

            color=color_map.get(
                algo,
                None
            ),

            linestyle=(
                "--"
                if algo.endswith("_false")
                else "-"
            ),

            linewidth=2,

            markersize=7,

            label=label_map.get(
                algo,
                algo
            )
        )


    plt.xlabel(
        "Network Load"
    )


    plt.ylabel(
        "Average FCT (s)"
    )


    plt.title(
        f"{benchmark}: "
        f"Pooled Average FCT - "
        f"Small Flows (<100 KB)"
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.legend()


    plt.tight_layout()


    out_path = (
        raw_graph_dir /
        f"{benchmark}_SMALL_average_FCT_v6_v7_v8.png"
    )


    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close()


    print(
        f"Saved: {out_path}"
    )


    # ========================================================
    # LARGE FLOWS
    # ========================================================

    plt.figure(
        figsize=(10, 7),
        dpi=120
    )


    for algo in required_algorithms:

        plt.plot(

            load_vals,

            large_raw_results[
                algo
            ],

            marker=marker_map.get(
                algo,
                "o"
            ),

            color=color_map.get(
                algo,
                None
            ),

            linestyle=(
                "--"
                if algo.endswith("_false")
                else "-"
            ),

            linewidth=2,

            markersize=7,

            label=label_map.get(
                algo,
                algo
            )
        )


    plt.xlabel(
        "Network Load"
    )


    plt.ylabel(
        "Average FCT (s)"
    )


    plt.title(
        f"{benchmark}: "
        f"Pooled Average FCT - "
        f"Large Flows (>1 MB)"
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.legend()


    plt.tight_layout()


    out_path = (
        raw_graph_dir /
        f"{benchmark}_LARGE_average_FCT_v6_v7_v8.png"
    )


    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close()


    print(
        f"Saved: {out_path}"
    )


# ============================================================
# NORMALIZED GRAPHS
#
# Saved in:
#
#     graph/
# ============================================================

print()
print("=" * 80)
print("GENERATING NORMALIZED GRAPHS")
print("=" * 80)


for benchmark in benchmarks:

    results = summary_results[
        benchmark
    ]


    load_vals = results[
        "loads"
    ]


    small_results = results[
        "small_normalized"
    ]


    large_results = results[
        "large_normalized"
    ]


    # ========================================================
    # SMALL FLOWS
    # ========================================================

    plt.figure(
        figsize=(10, 7),
        dpi=120
    )


    for algo in algorithms:

        plt.plot(

            load_vals,

            small_results[
                algo
            ],

            marker=marker_map.get(
                algo,
                "o"
            ),

            color=color_map.get(
                algo,
                None
            ),

            linestyle=(
                "--"
                if algo.endswith("_false")
                else "-"
            ),

            linewidth=2,

            markersize=7,

            label=(
                f"{label_map.get(algo, algo)} / ECMP"
            )
        )


    plt.axhline(

        y=1,

        linestyle="--",

        color="black",

        linewidth=1.5,

        label="ECMP baseline"
    )


    plt.xlabel(
        "Network Load"
    )


    plt.ylabel(
        "Normalized Average FCT"
    )


    plt.title(
        f"{benchmark}: "
        f"Pooled Average of v6, v7 and v8 - "
        f"Small Flows (<100 KB)"
    )


    plt.ylim(
        0,
        1.8
    )


    plt.yticks(
        np.arange(
            0,
            1.81,
            0.2
        )
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.legend()


    plt.tight_layout()


    out_path = (
        graph_dir /
        f"{benchmark}_SMALL_average_v6_v7_v8_relative_vs_load.png"
    )


    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close()


    print(
        f"Saved: {out_path}"
    )


    # ========================================================
    # LARGE FLOWS
    # ========================================================

    plt.figure(
        figsize=(10, 7),
        dpi=120
    )


    for algo in algorithms:

        plt.plot(

            load_vals,

            large_results[
                algo
            ],

            marker=marker_map.get(
                algo,
                "o"
            ),

            color=color_map.get(
                algo,
                None
            ),

            linestyle=(
                "--"
                if algo.endswith("_false")
                else "-"
            ),

            linewidth=2,

            markersize=7,

            label=(
                f"{label_map.get(algo, algo)} / ECMP"
            )
        )


    plt.axhline(

        y=1,

        linestyle="--",

        color="black",

        linewidth=1.5,

        label="ECMP baseline"
    )


    plt.xlabel(
        "Network Load"
    )


    plt.ylabel(
        "Normalized Average FCT"
    )


    plt.title(
        f"{benchmark}: "
        f"Pooled Average of v6, v7 and v8 - "
        f"Large Flows (>1 MB)"
    )


    plt.ylim(
        0,
        1.4
    )


    plt.yticks(
        np.arange(
            0,
            1.41,
            0.2
        )
    )


    plt.grid(
        True,
        alpha=0.3
    )


    plt.legend()


    plt.tight_layout()


    out_path = (
        graph_dir /
        f"{benchmark}_LARGE_average_v6_v7_v8_relative_vs_load.png"
    )


    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )


    plt.close()


    print(
        f"Saved: {out_path}"
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 80)
print("DONE")
print("=" * 80)

print()
print("Experiments pooled:")
print("    v6")
print("    v7")
print("    v8")

print()
print("Method:")
print("    Actual flows concatenated across v6/v7/v8")
print("    Then mean FCT calculated")

print()
print("Raw pooled FCT CSVs:")
print(f"    {save_dir}")

print()
print("Raw FCT graphs:")
print(f"    {raw_graph_dir}")

print()
print("Normalized graphs:")
print(f"    {graph_dir}")

print()
print("All processing completed successfully.")

EXPERIMENT DIRECTORIES
v6: OK
    /home/hsd/workspace/ns3-load-balance/rtt_sym_8_hosts_v6(prob1)/analysis_results
v7: OK
    /home/hsd/workspace/ns3-load-balance/rtt_sym_8_hosts_v7(prob1)/analysis_results
v8: OK
    /home/hsd/workspace/ns3-load-balance/rtt_sym_8_hosts_v8(prob1)/analysis_results


BENCHMARK: private_enterprise

--------------------------------------------------------------------------------
LOAD: 0.1
--------------------------------------------------------------------------------
v6: 5800 small, 15 large
v7: 5820 small, 10 large
v8: 5795 small, 17 large

Pooled SMALL flows: 17415
Pooled LARGE flows: 42

Pooled ECMP SMALL FCT: 0.003790 s
Pooled ECMP LARGE FCT: 0.347444 s

Normalized SMALL:
  Conga                              : 0.9883
  Weighted ECMP (Timeout)            : 0.9998
  Weighted ECMP (No Timeout)         : 0.9989
  Power-of-2 Random (Timeout)        : 0.9992
  Power-of-2 Random (No Timeout)     : 0.9980

Normalized LARGE:
  Conga                              